# Grokfast experiments

CSCI 567 final project: Vishal Patel, Salman Sattar, Tomohiko Uemura, Chinenye Okoro.

This notebook runs and analyzes every experiment in our study of
[*Grokfast: Accelerated Grokking by Amplifying Slow Gradients*](https://arxiv.org/abs/2405.20233)
on the modular-multiplication task (x·y mod 97). Training itself is the paper's own code
(`main.py`, with the filters in `grokfast.py`); this notebook defines the experiment grids,
runs them in parallel on one GPU, and turns the results into tables and plots.

[Open in Colab](https://colab.research.google.com/github/vip11017/csci567-grokfast/blob/salman/notebooks/grokfast_experiments.ipynb)
· use a **GPU runtime (T4 is enough)**: *Runtime → Change runtime type*.

| Section | What it does |
|---|---|
| 1. Setup | Clone the repo, check the GPU, choose where results are saved |
| 2. Experiment grids | Every configuration we run, by experiment |
| 3. Running grids | Background runner: `launch`, `status` |
| 4. Reading results | Load runs, steps to 95% validation accuracy, plots |
| 5. Replication | Baseline vs. Grokfast-MA vs. Grokfast-EMA, 3 seeds |
| 6. H2: weight decay | Does tuned weight decay alone close the gap? |
| 7. H1: momentum and frequency | Is it ordinary momentum, or the *slow* part of the gradient? |
| 8. H3: induced grokking | Draft: does it only help with tiny data and large initialization? |
| 9. Saving results | Keep results after the runtime shuts down |

Cells that start long GPU runs (`launch(...)`) are commented out, so *Run all* only sets up and
analyzes. Uncomment the one you want.

Background and write-ups live in `docs/` (start with `docs/ideas.md` and `docs/results-replication.md`).

## 1. Setup

Clones the `salman` branch (or pulls it if it's already here) and checks the GPU.

In [ ]:
import base64
import glob
import gzip
import json
import os
import re
import statistics
import subprocess
import sys
import threading
import time

import matplotlib.pyplot as plt
import torch

REPO_URL = "https://github.com/vip11017/csci567-grokfast.git"
BRANCH = "salman"
REPO_DIR = "/content/csci567-grokfast"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### Where results go

The Colab runtime is wiped when it shuts down. Set `USE_DRIVE = True` to write results
straight to Google Drive (Colab asks for permission once). Otherwise results stay in
`/content/results` until you save them (section 9).

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/csci567-grokfast"
else:
    RESULTS = "/content/results"
os.makedirs(RESULTS, exist_ok=True)
print("results go to", RESULTS)

## 2. Experiment grids

Each grid is a list of `main.py` runs. Every run stops once validation accuracy has held at
or above 95% for 1,000 steps, or at the step budget. Runs whose training accuracy is still
below 50% at step 20,000 are stopped and counted as failures: with strong weight decay
under Adam the model can't even fit the training set, and those runs would otherwise use
the whole budget.

Runs are ordered seed by seed, so a partly finished grid still covers every configuration
for seed 0.

In [ ]:
STOPPING = ["--stop_threshold", "0.95", "--stop_patience", "1000",
            "--abort_step", "20000", "--abort_train_acc", "0.5"]
MA = ["--filter", "ma", "--window_size", "100", "--lamb", "5.0"]
EMA = ["--alpha", "0.98", "--lamb", "2.0"]  # with --filter ema / ema_hp / ema_rand


def replication_grid(seed):
    """The paper's Table 3: baseline, Grokfast-MA, Grokfast-EMA."""
    yield f"rep_s{seed}_none", f"rep_s{seed}", []
    yield f"rep_s{seed}_ma", f"rep_s{seed}", MA + ["--weight_decay", "0.01"]
    yield f"rep_s{seed}_ema", f"rep_s{seed}", ["--filter", "ema"] + EMA + ["--weight_decay", "0.005"]


def h2_grid(seed):
    """Optimizer x weight decay x filter. Skips AdamW at 0 (same as Adam) and Adam/none/0
    (the replication baseline)."""
    for optimizer in ("Adam", "AdamW"):
        for filt in ("none", "ema"):  # slow no-filter runs first, for better packing
            for wd in (0, 0.001, 0.005, 0.01, 0.05, 0.1, 1.0):
                if wd == 0 and (optimizer == "AdamW" or filt == "none"):
                    continue
                args = ["--optimizer", optimizer, "--weight_decay", str(wd)]
                if filt == "ema":
                    args += ["--filter", "ema"] + EMA
                yield f"h2_s{seed}_{optimizer.lower()}_{filt}_wd{wd:g}", f"h2_s{seed}_{optimizer.lower()}", args


def h1_grid(seed):
    """Filters at the same gain, and more Adam momentum without a filter."""
    for wd in (0, 0.005):
        for filt in ("none", "ema", "ema_hp", "ema_rand"):
            args = ["--weight_decay", str(wd)]
            if filt != "none":
                args += ["--filter", filt] + EMA
            yield f"h1_s{seed}_b0.9_{filt}_wd{wd:g}", f"h1_s{seed}_b0.9", args
        for beta1 in (0.95, 0.98, 0.99):
            yield (f"h1_s{seed}_b{beta1}_none_wd{wd:g}", f"h1_s{seed}_b{beta1}",
                   ["--weight_decay", str(wd), "--beta1", str(beta1)])


MNIST_COMMON = ["--val_points", "10000", "--log_every", "200", "--optimization_steps", "100000",
                "--stop_threshold", "0.85", "--stop_patience", "1000",
                "--abort_step", "20000", "--abort_train_acc", "0.5"]


def h3_grid(seed):
    """DRAFT for the H3 owner: MNIST, train size x initialization scale x method.
    Methods: the paper's baseline (AdamW, weight decay 0.01), a no-filter run at the paper's
    Grokfast weight decay (2.0) to separate decay from filtering, and Grokfast-EMA as in the
    paper (alpha 0.8, lambda 0.1, weight decay 2.0)."""
    methods = {
        "base": ["--weight_decay", "0.01"],
        "wd2": ["--weight_decay", "2.0"],
        "ema": ["--weight_decay", "2.0", "--filter", "ema", "--alpha", "0.8", "--lamb", "0.1"],
    }
    for n in (1000, 10000, 50000):
        for init in (1, 3, 8):
            for tag, args in methods.items():
                label = f"h3_s{seed}_n{n}_i{init}_{tag}"
                yield (label, label, ["--train_points", str(n), "--initialization_scale", str(init)]
                       + MNIST_COMMON + args, "main_mnist.py")


def paper_ablations_grid(seed):
    """The paper's ablations with Grokfast-MA and no weight decay: Table 1 (two-stage),
    Figure 5 (slow component only) and Figure 4 (lambda and window sweeps)."""
    ma = lambda w, lamb: ["--filter", "ma", "--window_size", str(w), "--lamb", str(lamb)]
    slow = ["--filter", "ma_slow", "--window_size", "100", "--lamb", "5.0"]
    configs = {"ma": ma(100, 5), "ma2stage": ma(100, 5) + ["--two_stage"],
               "slow": slow, "slow2stage": slow + ["--two_stage"]}
    configs.update({f"lamb{lamb}": ma(100, lamb) for lamb in (1, 2, 10)})
    configs.update({f"win{w}": ma(w, 5) for w in (2, 5, 10, 20, 50, 200)})
    for tag, args in configs.items():
        yield f"abl_s{seed}_{tag}", f"abl_s{seed}_{tag}", args


# The other three datasets, as in the paper's README: its baseline, the README's "Grokfast"
# command as written (which never sets --filter, so only weight decay changes), and that
# command with the EMA filter actually on.
def paper_mnist_grid(seed):
    readme = ["--alpha", "0.8", "--lamb", "0.1", "--weight_decay", "2.0"]
    for tag, args in {"base": [], "readme": readme, "ema": readme + ["--filter", "ema"]}.items():
        yield f"mnist_s{seed}_{tag}", f"pm_s{seed}_{tag}", ["--log_every", "100"] + args, "main_mnist.py"


def paper_qm9_grid(seed):
    readme = ["--alpha", "0.9", "--lamb", "1.0", "--weight_decay", "0.01"]
    for tag, args in {"base": [], "readme": readme, "ema": readme + ["--filter", "ema"]}.items():
        yield f"qm9_s{seed}_{tag}", f"pq_s{seed}_{tag}_", ["--data_dir", QM9_DIR] + args, "main_qm9.py"


def paper_imdb_grid(seed):
    readme = ["--alpha", "0.98", "--lamb", "2.0", "--weight_decay", "10.0"]
    for tag, args in {"base": [], "readme": readme, "ema": readme + ["--filter", "ema"]}.items():
        yield f"imdb_s{seed}_{tag}", f"pi_s{seed}_{tag}_", ["--data_path", IMDB_CSV] + args, "main_imdb.py"


QM9_DIR = "/content/qm9_data"
IMDB_CSV = "/content/IMDB-Dataset.csv"

GRIDS = {"replication": replication_grid, "paper_ablations": paper_ablations_grid,
         "paper_mnist": paper_mnist_grid, "paper_qm9": paper_qm9_grid, "paper_imdb": paper_imdb_grid,
         "h2": h2_grid, "h1": h1_grid, "h3": h3_grid}


def build_runs(grid, seeds=(0, 1, 2), budget=100000):
    """Grids yield (name, label, args), or (name, label, args, script) for scripts other
    than main.py; those grids pass their own step budget and stopping options."""
    out = f"{RESULTS}/{grid}"
    runs = []
    for seed in seeds:
        for name, label, args, *script in GRIDS[grid](seed):
            script = script[0] if script else "main.py"
            common = ["--budget", str(budget)] + STOPPING if script == "main.py" else []
            cmd = [sys.executable, script, "--label", label, "--seed", str(seed),
                   "--out_dir", out] + common + args
            runs.append({"name": name, "cmd": cmd})
    return runs


for grid in GRIDS:
    print(f"{grid}: {len(build_runs(grid))} runs")

## 3. Running grids

`launch(grid)` runs a grid in a background thread, a fixed number of `main.py` processes at a
time, so the cell returns right away. On a T4, 8 at once works well: the model is tiny, so
most of the cost is Python overhead, not the GPU. A Grokfast run takes a few minutes; a
no-filter baseline can take an hour.

Each finished run leaves a `done_<name>.txt` marker and each started run a `pid_<name>.txt`,
so calling `launch` again (say, after a kernel restart) skips finished runs and picks up runs
that are still going instead of starting them twice. `status(grid)` shows progress.

In [ ]:
RUNNERS = {}


def _alive(pid):
    return os.path.exists(f"/proc/{pid}")


def _run_grid(grid, runs, parallel):
    out = f"{RESULTS}/{grid}"
    done = lambda name: os.path.exists(f"{out}/done_{name}.txt")
    queue = [r for r in runs if not done(r["name"])]
    running = {}
    for run in list(queue):  # adopt runs still going from an earlier kernel
        pid_file = f"{out}/pid_{run['name']}.txt"
        if os.path.exists(pid_file) and _alive(int(open(pid_file).read())):
            running[run["name"]] = int(open(pid_file).read())
            queue.remove(run)

    while queue or running:
        while queue and len(running) < parallel:
            run = queue.pop(0)
            with open(f"{out}/log_{run['name']}.txt", "w") as log:
                proc = subprocess.Popen(run["cmd"], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT,
                                        stdin=subprocess.DEVNULL, start_new_session=True)
            with open(f"{out}/pid_{run['name']}.txt", "w") as f:
                f.write(str(proc.pid))
            running[run["name"]] = proc

        for name, proc in list(running.items()):
            if isinstance(proc, subprocess.Popen):
                finished, rc = proc.poll() is not None, proc.returncode
            else:
                finished, rc = not _alive(proc), "unknown"
            if finished:
                with open(f"{out}/done_{name}.txt", "w") as f:
                    f.write(f"rc={rc}\n")
                del running[name]

        with open(f"{out}/status.json", "w") as f:
            json.dump({"grid": grid, "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
                       "total": len(runs), "done": sum(done(r["name"]) for r in runs),
                       "running": sorted(running), "queued": [r["name"] for r in queue]}, f, indent=1)
        time.sleep(10)


def launch(grid, parallel=8, seeds=(0, 1, 2), budget=100000):
    """Start (or resume) a grid in the background."""
    if grid in RUNNERS and RUNNERS[grid].is_alive():
        print(f"{grid} is already running")
        return
    os.makedirs(f"{RESULTS}/{grid}", exist_ok=True)
    runs = build_runs(grid, seeds, budget)
    RUNNERS[grid] = threading.Thread(target=_run_grid, args=(grid, runs, parallel), daemon=True)
    RUNNERS[grid].start()
    print(f"launched {grid}: {len(runs)} runs -> {RESULTS}/{grid}")


def status(grid):
    """How many runs are done, running and queued."""
    path = f"{RESULTS}/{grid}/status.json"
    if not os.path.exists(path):
        print(f"{grid}: not launched in this runtime")
        return
    with open(path) as f:
        s = json.load(f)
    print(f"{grid}: {s['done']}/{s['total']} done, {len(s['running'])} running, "
          f"{len(s['queued'])} queued (updated {s['updated']} UTC)")
    for name in s["running"]:
        print("  running:", name)

## 4. Reading results

`main.py` saves each run as `res_<name>.pt` with its training and validation curves (one point
every 10 steps). We report two numbers:

- **`first_95`**: the first step with validation accuracy ≥ 95%, as in the paper.
- **`held_95`**: the first step after which validation accuracy *stays* ≥ 95% for 1,000 steps.
  Many runs grok and then collapse briefly (validation accuracy can drop below 10% before
  recovering), so `first_95` alone flatters unstable settings.

In [ ]:
THRESHOLD = 0.95
HOLD_STEPS = 1000


def first_step_at(its, accs, threshold=THRESHOLD):
    return next((it for it, acc in zip(its, accs) if acc >= threshold), None)


def first_step_held(its, accs, threshold=THRESHOLD, hold=HOLD_STEPS):
    """First step from which accuracy stays >= threshold for `hold` steps."""
    run_start = None
    for it, acc in zip(its, accs):
        if acc < threshold:
            run_start = None
            continue
        run_start = it if run_start is None else run_start
        if it - run_start >= hold:
            return run_start
    return None


def load_runs(folder, prefix="res_", threshold=THRESHOLD):
    """All <prefix>*.pt files in a folder, keyed by run name. main.py writes res_*.pt,
    main_mnist.py writes mnist_*.pt. `first_95`/`held_95` use `threshold`."""
    runs = {}
    for path in sorted(glob.glob(f"{folder}/{prefix}*.pt")):
        r = torch.load(path, weights_only=True)
        r["first_95"] = first_step_at(r["its"], r["val_acc"], threshold)
        r["held_95"] = first_step_held(r["its"], r["val_acc"], threshold)
        runs[os.path.basename(path)[len(prefix):-3]] = r
    return runs


def summary(folder, match=""):
    """One line per run."""
    for name, r in load_runs(folder).items():
        if match in name:
            print(f"{name:45s} first95={r['first_95']}  last={r['its'][-1]}  val={r['val_acc'][-1]:.3f}")


def by_group(runs):
    """Group runs by everything after the seed tag: rep_s0_none -> 'none'."""
    groups = {}
    for name, r in runs.items():
        m = re.search(r"_s(\d+)_(.+)$", name)
        if m:
            groups.setdefault(m.group(2), {})[int(m.group(1))] = r
    return groups


def cell(seed_runs, metric="first_95"):
    """Median of a metric over seeds that reached it, with per-seed values."""
    hits = {s: r[metric] for s, r in sorted(seed_runs.items())}
    reached = [h for h in hits.values() if h is not None]
    per_seed = "/".join("-" if h is None else f"{h / 1000:.1f}k" for h in hits.values())
    median = f"{statistics.median(reached) / 1000:.1f}k" if 2 * len(reached) > len(hits) else "fail"
    return f"{median} ({per_seed})"


def plot_val_acc(runs, title, groups=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for color, (group, seed_runs) in zip(colors, sorted(by_group(runs).items())):
        if groups and group not in groups:
            continue
        for i, r in enumerate(seed_runs.values()):
            ax.plot(r["its"], r["val_acc"], color=color, lw=1, alpha=0.8, label=group if i == 0 else None)
    ax.axhline(THRESHOLD, color="grey", ls="--", lw=0.8)
    ax.set(xscale="log", xlabel="Optimization steps", ylabel="Validation accuracy", title=title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()

## 5. Replication

Before testing our own hypotheses we rerun every experiment in the paper with its own code and
settings, to check that the claims hold as published:

| | Paper's claim | Grid |
|---|---|---|
| 5.1 | Modular multiplication, ~50× fewer steps (Table 3) | `replication` |
| 5.2 | Ablations: two-stage training, slow component only, λ and window sweeps (Table 1, Figures 4–5) | `paper_ablations` |
| 5.3 | MNIST, 22× fewer steps, final accuracy 89.8% → 91.2% (Table 4) | `paper_mnist` |
| 5.4 | QM9, minimum validation loss 0.00659 → 0.00348 (Table 5) | `paper_qm9` |
| 5.5 | IMDb, best validation accuracy 0.84 → 0.90 (Table 6) | `paper_imdb` |

For MNIST, QM9 and IMDb the README's "Grokfast" command never sets `--filter`, and the scripts
default to no filter, so as written it only changes weight decay. We run three versions of each:
the paper's baseline (`base`), the README's command as written (`readme`), and that command with
the EMA filter on (`ema`).

### 5.1 Table 3: modular multiplication

The paper's headline comparison (Table 3): baseline Adam with no weight decay, Grokfast-MA
(window 100, λ 5, weight decay 0.01) and Grokfast-EMA (α 0.98, λ 2, weight decay 0.005).
Paper: 39,890 / 790 / 910 steps to 95% validation accuracy.

We ran this on 2026-10-30 (3 seeds, Colab T4); the saved runs are in `results/replication/`
and the write-up is `docs/results-replication.md`. To rerun it, uncomment `launch` (about
1.5 hours on a T4, mostly the baseline runs) and point `folder` at `f"{RESULTS}/replication"`.

In [ ]:
# launch("replication")
folder = f"{REPO_DIR}/results/replication"
replication = load_runs(folder)
print(f"{'':25s} {'first_95':24s} held_95")
for group, seed_runs in sorted(by_group(replication).items()):
    print(f"{group:25s} {cell(seed_runs):24s} {cell(seed_runs, 'held_95')}")
plot_val_acc(replication, "Replication, 3 seeds")

### 5.2 Ablations (Table 1, Figures 4 and 5)

All with Grokfast-MA and no weight decay, 3 seeds (39 runs). Paper values, steps to 95%:

- **Table 1:** baseline 39,890; Grokfast-MA from the start (`ma`) 2,940; MA only after step 500
  (`ma2stage`) 1,920; slow component only after step 500 (`slow2stage`) never converges.
- **Figure 5:** using *only* the averaged gradient (`slow`) is slower and unstable.
- **Figure 4:** best at λ = 5 and window 100, with worse results on either side
  (`lamb1/2/10` at window 100, `win2 … 200` at λ 5).

The "slow component only" filter isn't in the paper's released code; it's
`gradfilter_ma_slow` in `grokfast.py`, following the paper's description.

In [ ]:
# launch("paper_ablations")   # uncomment to run (mostly short runs)

In [ ]:
PAPER_ABLATIONS = {"ma": 2940, "ma2stage": 1920, "slow2stage": "not converged"}
ablations = load_runs(f"{RESULTS}/paper_ablations")
print(f"{'':30s} {'first_95':24s} {'held_95':24s} paper")
for group, seed_runs in sorted(by_group(ablations).items()):
    tag = group.split("_")[0]
    print(f"{group:30s} {cell(seed_runs):24s} {cell(seed_runs, 'held_95'):24s} {PAPER_ABLATIONS.get(tag, '')}")

### 5.3 MNIST (Table 4)

3-layer MLP on 1,000 training images, weights initialized 8× larger, 100,000 steps, as in the
paper. Validation is the MNIST test set (the paper's setup). Paper: steps to the threshold
44,022 → 2,001 (22×), final accuracy 89.8% → 91.2%. The paper's threshold is unclear (95% in
Table 4, 85% in Figure 9, and accuracy never reaches 95%), so we report both.

In [ ]:
# launch("paper_mnist")   # uncomment to run (~30 min per run on a T4)

In [ ]:
def load_raw(folder, prefix):
    """Result files from main_mnist.py / main_qm9.py / main_imdb.py, keyed by run name."""
    return {os.path.basename(p)[len(prefix):-3]: torch.load(p, weights_only=False)
            for p in sorted(glob.glob(f"{folder}/{prefix}*.pt"))}


def by_tag(runs):
    """Group paper_* runs by their method tag (base / readme / ema) across seeds."""
    groups = {}
    for name, r in runs.items():
        m = re.search(r"_s(\d+)_([a-z]+)", name)
        if m:
            groups.setdefault(m.group(2), {})[int(m.group(1))] = r
    return groups


def median_str(values, fmt="{:.4f}"):
    values = [v for v in values if v is not None]
    return fmt.format(statistics.median(values)) if values else "-"


mnist = load_raw(f"{RESULTS}/paper_mnist", "mnist_")
print(f"{'':8s} {'first 85%':>10s} {'first 95%':>10s} {'final val acc':>14s}   paper: 44,022 -> 2,001 steps, 89.8% -> 91.2%")
for tag, seed_runs in by_tag(mnist).items():
    rs = list(seed_runs.values())
    print(f"{tag:8s} {median_str([first_step_at(r['its'], r['val_acc'], 0.85) for r in rs], '{:.0f}'):>10s} "
          f"{median_str([first_step_at(r['its'], r['val_acc'], 0.95) for r in rs], '{:.0f}'):>10s} "
          f"{median_str([r['val_acc'][-1] for r in rs]):>14s}")

### 5.4 QM9 (Table 5)

Graph network predicting molecular polarizability from 100 molecules (50 train / 50 validation),
weights initialized 3× larger. Paper: minimum validation loss 0.00659 → 0.00348.

QM9 is downloaded once before launching, so parallel runs don't all download it at the same time.

In [ ]:
!pip -q install torch_geometric
from torch_geometric.datasets import QM9
QM9(QM9_DIR)  # downloads and preprocesses once
# launch("paper_qm9")   # uncomment to run

In [ ]:
qm9 = load_raw(f"{RESULTS}/paper_qm9", "qm9_")
print(f"{'':8s} {'min val loss':>13s}   paper: 0.00659 -> 0.00348")
for tag, seed_runs in by_tag(qm9).items():
    print(f"{tag:8s} {median_str([float(r['val_best']) for r in seed_runs.values()], '{:.5f}'):>13s}")

### 5.5 IMDb (Table 6)

2-layer LSTM on 1,000 reviews (75/25 split), weights initialized 6× larger, 10,000 steps.
Paper: best validation accuracy 0.84 → 0.90, minimum validation loss 0.517 → 0.412.
The dataset CSV is downloaded from the link in the paper's README.

In [ ]:
!gdown -q 11dNiqE5Y5tigaeyYP6JPV3xxa1MOu1L- -O {IMDB_CSV}
# launch("paper_imdb")   # uncomment to run

In [ ]:
imdb = load_raw(f"{RESULTS}/paper_imdb", "imdb_")
print(f"{'':8s} {'best val acc':>13s} {'min val loss':>13s}   paper: 0.84 -> 0.90, 0.517 -> 0.412")
for tag, seed_runs in by_tag(imdb).items():
    rs = list(seed_runs.values())
    print(f"{tag:8s} {median_str([max(r['val_acc']) for r in rs], '{:.3f}'):>13s} "
          f"{median_str([min(r['val_loss']) for r in rs], '{:.3f}'):>13s}")

## 6. H2: weight decay vs. the filter

The paper's baseline uses no weight decay, while its Grokfast runs use 0.005–0.01. This grid
asks whether tuned weight decay alone explains most of the speedup:

- optimizer: Adam (weight decay added to the gradient, as in the paper's `main.py`) and AdamW (decoupled)
- weight decay: 0, 0.001, 0.005, 0.01, 0.05, 0.1, 1.0
- filter: none, Grokfast-EMA (α 0.98, λ 2)
- seeds 0–2, 75 runs, about 5 hours on a T4

Note that Grokfast amplifies the loss gradient *before* Adam adds the weight-decay term, so at
the same nominal weight decay the filter makes decay relatively weaker.

In [ ]:
# launch("h2")   # done on 2026-10-30; runs are in results/h2/ (uncomment to rerun, ~5 h on a T4)

In [ ]:
status("h2")

In [ ]:
h2 = load_runs(f"{RESULTS}/h2")
groups = by_group(h2)
groups["adam_none"] = by_group(replication)["none"]  # weight decay 0 baseline, from section 5

columns = [("adam_none", "Adam, no filter"), ("adam_ema_a0980_l2", "Adam + EMA"),
           ("adamw_none", "AdamW, no filter"), ("adamw_ema_a0980_l2", "AdamW + EMA")]
suffix = {0: "", 0.001: "_wd10e-03", 0.005: "_wd50e-03", 0.01: "_wd10e-02",
          0.05: "_wd50e-02", 0.1: "_wd10e-01", 1.0: "_wd10e+00"}

for metric, what in [("first_95", "first reach 95%"), ("held_95", "stay at >= 95% for 1,000 steps")]:
    print(f"\nSteps until val accuracy {what}: median (seed 0/1/2)\n")
    print(f"{'weight decay':>12} | " + " | ".join(f"{title:24s}" for _, title in columns))
    for wd, s in suffix.items():
        row = []
        for key, _ in columns:
            if wd == 0 and key.startswith("adamw"):
                row.append("(same as Adam)")
            else:
                seed_runs = groups.get(key + s)
                row.append(cell(seed_runs, metric) if seed_runs else "")
        print(f"{wd:>12g} | " + " | ".join(f"{c:24s}" for c in row))

In [ ]:
# Steps to 95% vs. weight decay. Lines are medians over seeds; dots are single seeds.
# A missing point means fewer than half the seeds reached 95% (the model couldn't fit).
wds = [wd for wd in suffix if wd > 0]
fig, ax = plt.subplots(figsize=(7, 4.2))
for (key, title), color in zip(columns, plt.rcParams["axes.prop_cycle"].by_key()["color"]):
    medians = []
    for wd in wds:
        seed_runs = groups.get(key + suffix[wd], {})
        hits = [r["first_95"] for r in seed_runs.values() if r["first_95"] is not None]
        ax.scatter([wd] * len(hits), hits, color=color, alpha=0.35, s=12)
        medians.append(statistics.median(hits) if 2 * len(hits) > len(seed_runs) else float("nan"))
    ax.plot(wds, medians, "o-", color=color, label=title)
baseline = statistics.median(r["first_95"] for r in groups["adam_none"].values())
ax.axhline(baseline, color="grey", ls=":", lw=1, label="paper's baseline (Adam, no filter, no decay)")
ax.set(xscale="log", yscale="log", xlabel="weight decay", ylabel="steps to 95% val accuracy",
       title="H2: steps to grok vs. weight decay (3 seeds)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
plt.show()

## 7. H1: ordinary momentum, or the slow part of the gradient?

Adam already averages gradients (β₁ = 0.9); Grokfast-EMA adds a second, slower average in
front of it. This grid separates the two explanations, with Adam at weight decay 0 and 0.005:

- **filters at the same gain (λ 2, α 0.98):** none, EMA (Grokfast), high-pass EMA
  `g + λ(g − EMA(g))`, and a random direction with the same size as `λ·EMA(g)`
- **more Adam momentum, no filter:** β₁ ∈ {0.95, 0.98, 0.99}
- seeds 0–2, 42 runs

If the slow component is what matters, Grokfast should beat both the high-pass and random
filters, and higher β₁ shouldn't match it.

In [ ]:
# launch("h1")   # uncomment to run (~4 h on a T4)

In [ ]:
status("h1")

In [ ]:
h1 = load_runs(f"{RESULTS}/h1")
for group, seed_runs in sorted(by_group(h1).items()):
    print(f"{group:35s} {cell(seed_runs)}")

## 8. H3: does Grokfast only help in artificial grokking setups?

**Status: not started. This section is a draft for whoever takes H3.**

The paper's MNIST experiment forces grokking with a tiny training set (1,000 images) and
weights initialized 8× larger than usual. H3 asks whether Grokfast still helps when those
conditions are relaxed: training-set size {1k, 10k, 50k} × initialization scale {1, 3, 8}.

The draft grid (`h3_grid` in section 2) runs three methods per setting, with 3 seeds (81 runs):

- `base`: the paper's MNIST baseline (AdamW, weight decay 0.01, no filter)
- `wd2`: no filter at weight decay 2.0, the decay the paper uses *with* Grokfast. The paper
  compares Grokfast at 2.0 against a baseline at 0.01, the same confound as in H2.
- `ema`: Grokfast-EMA with the paper's MNIST settings (α 0.8, λ 0.1, weight decay 2.0). Note
  that the README's MNIST command never actually sets `--filter ema`; the grid does.

It uses the options added to `main_mnist.py`:

- `--val_points 10000` holds out the last 10,000 training images for validation, and the test
  set is scored once at the end (`test_acc` in the results file). Without it, the script uses
  the test set as its validation set, as in the paper.
- `--stop_threshold 0.85`: the paper's MNIST threshold is ambiguous (95% in Table 4, 85% in
  Figure 9, and final accuracy is about 91%), so the draft uses an explicit 85%.
- `--log_every 200` gives finer steps-to-threshold than the default (about 150 evaluations
  per run). Each evaluation scores the whole training set, which is slow at 50k images.

**Decisions for the H3 owner:** the threshold, the step budget, whether 3 methods are the
right ones, and whether to tune the learning rate. Try one seed first (`seeds=(0,)`).

In [ ]:
# launch("h3", seeds=(0,))   # uncomment to run; start with one seed

In [ ]:
status("h3")

In [ ]:
h3 = load_runs(f"{RESULTS}/h3", prefix="mnist_", threshold=0.85)
print(f"{'':32s} {'first 85%':24s} {'held 85%':24s} test acc")
for group, seed_runs in sorted(by_group(h3).items()):
    test = [r["test_acc"] for r in seed_runs.values() if "test_acc" in r]
    test_str = f"{statistics.median(test):.3f}" if test else "-"
    print(f"{group:32s} {cell(seed_runs):24s} {cell(seed_runs, 'held_95'):24s} {test_str}")

## 9. Saving results

If `USE_DRIVE` was off, copy everything to Drive before the runtime shuts down:

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/csci567-grokfast && cp -r {RESULTS}/* /content/drive/MyDrive/csci567-grokfast/

Without Drive, `export(grid)` prints a compressed copy of every run's curves (downsampled to
at most 300 points) that can be pasted into a text file and turned back into `res_<name>.json`
files with `decode`.

In [ ]:
def export(grid, match="", max_points=300):
    runs = {}
    for name, r in load_runs(f"{RESULTS}/{grid}").items():
        if match not in name:
            continue
        step = max(1, len(r["its"]) // max_points)
        keep = sorted(set(range(0, len(r["its"]), step)) | {len(r["its"]) - 1})
        runs[name] = {"first_95": r["first_95"], **{k: [r[k][i] for i in keep]
                      for k in ("its", "train_acc", "val_acc", "train_loss", "val_loss")}}
    print(base64.b64encode(gzip.compress(json.dumps(runs).encode())).decode())


def decode(blob_file, out_dir):
    runs = json.loads(gzip.decompress(base64.b64decode(open(blob_file).read().strip())))
    os.makedirs(out_dir, exist_ok=True)
    for name, run in runs.items():
        with open(f"{out_dir}/res_{name}.json", "w") as f:
            json.dump(run, f)
    print(f"wrote {len(runs)} runs to {out_dir}")